In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv('../data/processed/orders.csv')

orders.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714
3,4,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310
4,5,US-2016-108966,10/11/2016,10/18/2016,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311.0,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164


In [3]:
orders['order_date'].dtype

dtype('O')

In [4]:
orders['order_date'] = pd.to_datetime(orders['order_date'])


In [6]:
orders['order_date'].dtype

dtype('<M8[ns]')

In [7]:
orders['month'] = orders['order_date'].dt.to_period('M')

In [8]:
orders[['order_date', 'month']].head(10)

,order_date,month
0,2017-11-08,2017-11
1,2017-11-08,2017-11
2,2017-06-12,2017-06
3,2016-10-11,2016-10
4,2016-10-11,2016-10
5,2015-06-09,2015-06
6,2015-06-09,2015-06
7,2015-06-09,2015-06
8,2015-06-09,2015-06
9,2015-06-09,2015-06


In [9]:
customer_monthly = (orders.groupby(['customer_id', 'month'])['sales'].sum().reset_index())

In [10]:
customer_monthly.head(10)

,customer_id,month,sales
0,AA-10315,2015-03,726.548
1,AA-10315,2015-09,29.500
2,AA-10315,2016-10,26.960
3,AA-10315,2017-03,4406.072
4,AA-10315,2018-06,374.480
5,AA-10375,2015-04,16.520
6,AA-10375,2015-10,34.272
7,AA-10375,2016-02,178.370
8,AA-10375,2016-05,5.248
9,AA-10375,2016-11,84.960


In [12]:
customer_monthly = (orders.groupby(['customer_id', 'month']).agg(total_sales = ('sales', 'sum'),total_profit = ('profit', 'sum')).reset_index())

In [13]:
customer_monthly.head(10)

,customer_id,month,total_sales,total_profit
0,AA-10315,2015-03,726.548,267.4224
1,AA-10315,2015-09,29.500,13.2826
2,AA-10315,2016-10,26.960,7.0096
3,AA-10315,2017-03,4406.072,-747.1021
4,AA-10315,2018-06,374.480,96.5050
5,AA-10375,2015-04,16.520,5.5755
6,AA-10375,2015-10,34.272,11.1384
7,AA-10375,2016-02,178.370,62.0658
8,AA-10375,2016-05,5.248,0.5904
9,AA-10375,2016-11,84.960,6.3720


In [15]:
customer_monthly = (orders.groupby(['customer_id', 'month']).agg(total_sales = ('sales', 'sum'), total_profit = ('profit', 'sum'), total_orders = ('order_id', 'nunique')).reset_index())
customer_monthly.head(10)

,customer_id,month,total_sales,total_profit,total_orders
0,AA-10315,2015-03,726.548,267.4224,1
1,AA-10315,2015-09,29.500,13.2826,1
2,AA-10315,2016-10,26.960,7.0096,1
3,AA-10315,2017-03,4406.072,-747.1021,1
4,AA-10315,2018-06,374.480,96.5050,1
5,AA-10375,2015-04,16.520,5.5755,1
6,AA-10375,2015-10,34.272,11.1384,1
7,AA-10375,2016-02,178.370,62.0658,1
8,AA-10375,2016-05,5.248,0.5904,1
9,AA-10375,2016-11,84.960,6.3720,1


In [16]:
customer_monthly[customer_monthly['total_orders']>1].head(10)

,customer_id,month,total_sales,total_profit,total_orders
33,AB-10060,2018-11,2835.538,441.7752,2
63,AB-10255,2018-06,258.690,107.6623,2
99,AF-10870,2017-05,1591.470,348.2534,2
101,AF-10870,2018-11,280.386,-38.3984,2
116,AG-10330,2018-12,412.152,-66.8801,4
124,AG-10495,2016-10,551.177,-64.6421,3
130,AG-10525,2016-09,701.392,-59.2320,2
136,AG-10675,2018-09,92.480,29.7663,2
142,AG-10900,2016-09,911.179,-25.0458,2
143,AG-10900,2016-12,135.432,43.8728,2


In [19]:
monthly_summary = (orders.groupby(['month']).agg(total_sales = ('sales', 'sum'), total_profit = ('profit', 'sum'),total_orders = ('order_id', 'nunique')).reset_index())
monthly_summary.head(10)

,month,total_sales,total_profit,total_orders
0,2015-01,14236.8950,2450.1907,32
1,2015-02,4519.8920,862.3084,28
2,2015-03,55691.0090,498.7299,71
3,2015-04,28295.3450,3488.8352,66
4,2015-05,23648.2870,2738.7096,69
5,2015-06,34595.1276,4976.5244,66
6,2015-07,33946.3930,-841.4826,65
7,2015-08,27909.4685,5318.1050,72
8,2015-09,81777.3508,8328.0994,130
9,2015-10,31453.3930,3448.2573,78
